In [1]:
# Cell 1 — Setup, imports, Google Drive (identical to Step 8)

from __future__ import annotations

import os
import gc
import json
import math
import time
import random
import hashlib
import shutil
import zipfile
import warnings
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Any, Iterable

import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.metrics import (
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    precision_recall_curve,
    roc_curve,
)
from sklearn.preprocessing import MinMaxScaler

try:
    from IPython.display import display
except Exception:
    display = print

warnings.filterwarnings("ignore")

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    print("Không chạy trong Google Colab; bỏ qua mount Drive.")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE} | PyTorch: {torch.__version__}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {torch.cuda.get_device_name(0)} | VRAM: {props.total_memory / 1e9:.1f} GB")


Mounted at /content/drive
Device: cuda | PyTorch: 2.11.0+cu128
GPU: NVIDIA A100-SXM4-40GB | VRAM: 42.4 GB


In [2]:
# Cell 2 — Master configuration

RUN_MODE = "all"  # rerun | aggregate | all
VALID_RUN_MODES = {"rerun", "aggregate", "all"}
if RUN_MODE not in VALID_RUN_MODES:
    raise ValueError(f"RUN_MODE must be one of {sorted(VALID_RUN_MODES)}")

# ---- Step 9 experiment switch ----------------------------------------------
# baseline : reproduce the published numbers (context target NOT detached)
# detach   : stop-gradient on the context target, rho unchanged  <-- supervisor ask
# drop     : rho = 0, context reconstruction term removed entirely
ARMS_TO_RUN = ["detach", "drop"]
VALID_ARMS = {"baseline", "detach", "drop"}
if not set(ARMS_TO_RUN) <= VALID_ARMS:
    raise ValueError(f"ARMS_TO_RUN must be a subset of {sorted(VALID_ARMS)}")

# set by run_context_target_phase(); Cell 6 reads it at every forward pass
CONTEXT_TARGET_MODE = ARMS_TO_RUN[0]

DATASETS_TO_RUN = ["CIC_V2", "UNSW_V2"]
MODEL_SEEDS = [42, 123, 456]
BENCHMARK_SPLIT_SEED = 2027
PROTOCOL_NAME = "exact_feature_group"

DRIVE_RUN_ROOT = Path("/content/drive/MyDrive/THAT_IDS_RERUN")
STEP4_ROOT = DRIVE_RUN_ROOT / "conference_step4_exact_feature_group_v1"
STEP4_SHARED_CACHE_ROOT = DRIVE_RUN_ROOT / "conference_step4_shared_feature_cache_v1"
STEP5_ROOT = DRIVE_RUN_ROOT / "conference_step5_literature_subset_v1"
STEP5_CONFIRMATION_ROOT = STEP5_ROOT / "confirmation"
STEP5_SUBSET_DEFINITION_ROOT = STEP5_ROOT / "subset_definition"
STEP5_SUBSET_CACHE_ROOT = STEP5_ROOT / "subset_feature_cache"
STEP6_ROOT = DRIVE_RUN_ROOT / "conference_step6_public_baselines_v1"
STEP7_ROOT = DRIVE_RUN_ROOT / "conference_step7_tabular_loo_v1"

STEP8_ROOT = DRIVE_RUN_ROOT / "conference_step8_supervisor_revision_master_v1"

STEP9_EXPERIMENT_TAG = "conference_step9_context_target_v1"
OUTPUT_ROOT = DRIVE_RUN_ROOT / STEP9_EXPERIMENT_TAG
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
AUDIT_ROOT = OUTPUT_ROOT / "audit"
AUDIT_ROOT.mkdir(parents=True, exist_ok=True)

DATA_PATH_CANDIDATES = {
    "UNSW_V2": [
        "/content/drive/MyDrive/datasets/NF-UNSW-NB15-v2.csv",
        "/content/drive/MyDrive/NF-UNSW-NB15-v2.csv",
    ],
    "CIC_V2": [
        "/content/drive/MyDrive/datasets/NF-CSE-CIC-IDS2018-v2.csv",
        "/content/drive/MyDrive/NF-CSE-CIC-IDS2018-v2.csv",
    ],
}

# Step 9 trains only the Full variant on the frozen Step-5 rows and split.
BATCH_PARTITION_SEEDS = [0]
RELATION_VARIANTS_TO_RUN = []
MAX_EPOCHS = 80
PATIENCE = 12
SAVE_LAST_EVERY = 1
WINDOW_SIZE = 5000

# Model constants: exact Step-5 architecture
HIDDEN_DIM = 96
AE_HIDDEN = 192
LATENT_DIM = 96
GRAPH_DROPOUT = 0.10
ENC_DROPOUT = 0.10
CLS_DROPOUT = 0.15
INIT_GATE_LOGIT = -1.5
CTX_RECON_WEIGHT = 0.10

LEARNING_RATE = 5e-5
WEIGHT_DECAY = 1e-5
GRAD_CLIP_NORM = 1.0
LAMBDA_RECON = 1.0
LAMBDA_CLS = 1.0
FOCAL_ALPHA = 0.75
FOCAL_GAMMA = 2.0
POS_WEIGHT_CAP = 12.0
SCORE_CLS_WEIGHT = 0.90
SCORE_RECON_WEIGHT = 0.10
CLIP_MIN, CLIP_MAX = -10.0, 10.0

# Audit / post-processing
CSV_CHUNK_SIZE = 500_000
BETA_VALUES = [0.0, 0.25, 0.50, 0.75, 0.90, 1.0]
FPR_TARGETS = [1e-4, 5e-4, 1e-3, 5e-3, 1e-2]
RECALL_TARGETS = [0.90, 0.95, 0.99]
TEST_CURVE_FPR_TARGETS = FPR_TARGETS
TEST_CURVE_RECALL_TARGETS = RECALL_TARGETS
BOOTSTRAP_REPLICATES = 1000
BOOTSTRAP_SEED = 2027
RUN_GROUP_BOOTSTRAP = True
HASH_VERIFY_MODE = "sample"  # none | sample | full
HASH_VERIFY_SAMPLE_GROUPS = 10000
STRICT_PREFLIGHT = True

# Optional fast debug
SMOKE_MODE = False
SMOKE_MAX_ROWS_PER_SPLIT = 20000
if SMOKE_MODE:
    MAX_EPOCHS = 2
    PATIENCE = 2
    MODEL_SEEDS = [42]
    BOOTSTRAP_REPLICATES = 50

# Published Step-5/8 gate values, used only for the comparison printout.
BASELINE_GATE_REFERENCE = {
    "CIC_V2": [0.185163, 0.185217, 0.184949],
    "UNSW_V2": [0.184123, 0.184319, 0.184362],
}
INIT_GATE_VALUE = 1.0 / (1.0 + math.exp(-INIT_GATE_LOGIT))  # 0.182426

print("Step 9 output:", OUTPUT_ROOT)
print("RUN_MODE:", RUN_MODE)
print("Arms:", ARMS_TO_RUN)
print("Datasets:", DATASETS_TO_RUN)
print("Model seeds:", MODEL_SEEDS)
print(f"Gate initialization: theta0={INIT_GATE_LOGIT} -> g0={INIT_GATE_VALUE:.6f}")
print(f"Total runs planned: {len(ARMS_TO_RUN) * len(DATASETS_TO_RUN) * len(MODEL_SEEDS)}")


Step 9 output: /content/drive/MyDrive/THAT_IDS_RERUN/conference_step9_context_target_v1
RUN_MODE: all
Arms: ['detach', 'drop']
Datasets: ['CIC_V2', 'UNSW_V2']
Model seeds: [42, 123, 456]
Gate initialization: theta0=-1.5 -> g0=0.182426
Total runs planned: 12


In [3]:
# Cell 3 — Paths, fingerprints, atomic files, RNG save/restore

def resolve_dataset_path(dataset_name: str) -> str:
    for candidate in DATA_PATH_CANDIDATES.get(dataset_name, []):
        if Path(candidate).exists():
            return candidate
    raise FileNotFoundError(
        "Không tìm thấy dataset " + dataset_name + ". Đã thử:\n"
        + "\n".join(DATA_PATH_CANDIDATES.get(dataset_name, []))
    )

def step5_dataset_dir(dataset_name: str) -> Path:
    return STEP5_CONFIRMATION_ROOT / dataset_name

def step5_shared_dir(dataset_name: str) -> Path:
    return (
        step5_dataset_dir(dataset_name)
        / PROTOCOL_NAME / "shared" / f"seed_{BENCHMARK_SPLIT_SEED}"
    )

def subset_definition_dir(dataset_name: str) -> Path:
    return STEP5_SUBSET_DEFINITION_ROOT / dataset_name

def step5_full_task_dir(dataset_name: str, seed: int) -> Path:
    return step5_dataset_dir(dataset_name) / PROTOCOL_NAME / "full" / f"seed_{seed}"

def step7_task_dir(dataset_name: str, model_name: str, seed: int) -> Path:
    return STEP7_ROOT / dataset_name / PROTOCOL_NAME / model_name / f"seed_{seed}"

def batch_task_dir(dataset_name: str, partition_seed: int, model_seed: int) -> Path:
    return (
        OUTPUT_ROOT / "batch_partition_sensitivity" / dataset_name
        / f"partition_{partition_seed}" / "full" / f"seed_{model_seed}"
    )

def relation_task_dir(dataset_name: str, variant_name: str, model_seed: int) -> Path:
    return (
        OUTPUT_ROOT / "relation_ablation" / dataset_name
        / variant_name / f"seed_{model_seed}"
    )

def load_json(path: Path) -> Dict[str, Any]:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def atomic_json_dump(obj: Any, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)
    os.replace(tmp, path)

def atomic_torch_save(obj: Any, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    torch.save(obj, tmp)
    os.replace(tmp, path)

def load_torch_checkpoint(path: Path, map_location: torch.device) -> Dict[str, Any]:
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)

def config_fingerprint(payload: Dict[str, Any]) -> str:
    return hashlib.sha256(
        json.dumps(payload, sort_keys=True, default=str).encode("utf-8")
    ).hexdigest()[:24]

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def capture_rng_state() -> Dict[str, Any]:
    state = {
        "python": random.getstate(),
        "numpy": np.random.get_state(),
        "torch": torch.get_rng_state(),
    }
    if torch.cuda.is_available():
        state["cuda"] = torch.cuda.get_rng_state_all()
    return state

def _cpu_byte_tensor(value: Any) -> torch.Tensor:
    if isinstance(value, torch.Tensor):
        return value.detach().to(device="cpu", dtype=torch.uint8).contiguous()
    return torch.as_tensor(value, dtype=torch.uint8, device="cpu").contiguous()

def restore_rng_state(state: Optional[Dict[str, Any]]) -> None:
    if not state:
        return
    if "python" in state:
        random.setstate(state["python"])
    if "numpy" in state:
        np.random.set_state(state["numpy"])
    if "torch" in state:
        torch.set_rng_state(_cpu_byte_tensor(state["torch"]))
    if torch.cuda.is_available() and "cuda" in state:
        values = state["cuda"]
        if isinstance(values, torch.Tensor):
            values = [values]
        for device_index, cuda_state in enumerate(list(values)[:torch.cuda.device_count()]):
            torch.cuda.set_rng_state(_cpu_byte_tensor(cuda_state), device=device_index)

def update_master_progress(phase: str, dataset: Optional[str] = None, status: str = "completed",
                           details: Optional[Dict[str, Any]] = None) -> None:
    path = OUTPUT_ROOT / "master_progress.json"
    progress = load_json(path) if path.exists() else {
        "experiment_tag": STEP9_EXPERIMENT_TAG,
        "created_at_unix": time.time(),
        "phases": {},
    }
    key = phase if dataset is None else f"{phase}:{dataset}"
    progress["phases"][key] = {
        "status": status,
        "updated_at_unix": time.time(),
        "details": details or {},
    }
    atomic_json_dump(progress, path)

def find_first_existing(candidates: Iterable[Path]) -> Optional[Path]:
    for path in candidates:
        if path.exists():
            return path
    return None

def recursive_find_prediction(root: Path, dataset: str, model_tokens: List[str], seed: int) -> Optional[Path]:
    if not root.exists():
        return None
    seed_token = f"seed_{seed}"
    matches = []
    for path in root.rglob("test_predictions.npz"):
        text = str(path).lower()
        if dataset.lower() in text and seed_token.lower() in text and all(t.lower() in text for t in model_tokens):
            matches.append(path)
    return sorted(matches, key=lambda p: len(str(p)))[0] if matches else None


def context_task_dir(arm: str, dataset_name: str, model_seed: int) -> Path:
    return OUTPUT_ROOT / "runs" / arm / dataset_name / f"seed_{model_seed}"


In [4]:
# Cell 4 — Load exact frozen Step-5 subset, split and scaler

@dataclass
class FixedSubsetBundle:
    name: str
    feature_names: List[str]
    X_raw: np.ndarray
    src_ip: np.ndarray
    dst_ip: np.ndarray
    protocol: np.ndarray
    labels: np.ndarray
    attack_codes: np.ndarray
    attack_classes: np.ndarray
    subset_global_indices: np.ndarray
    subset_fingerprint: str

def read_step5_feature_names(dataset_name: str) -> List[str]:
    path = step5_dataset_dir(dataset_name) / "model_feature_names.csv"
    table = pd.read_csv(path)
    return table.sort_values("feature_index")["feature"].astype(str).tolist()

def preflight_step5(dataset_name: str) -> None:
    required = [
        subset_definition_dir(dataset_name) / "subset_indices_int32.npy",
        subset_definition_dir(dataset_name) / "subset_manifest.json",
        step5_dataset_dir(dataset_name) / "model_feature_names.csv",
        step5_shared_dir(dataset_name) / "split_indices.npz",
        step5_shared_dir(dataset_name) / "split_metadata.json",
        step5_shared_dir(dataset_name) / "split_overlap_audit.json",
        step5_shared_dir(dataset_name) / "scaler.joblib",
    ]
    for seed in MODEL_SEEDS:
        required.extend([
            step5_full_task_dir(dataset_name, seed) / "best.pt",
            step5_full_task_dir(dataset_name, seed) / "test_metrics.json",
            step5_full_task_dir(dataset_name, seed) / "test_predictions.npz",
        ])
    missing = [str(p) for p in required if not p.exists()]
    if missing:
        message = "Thiếu Step-5 artifacts:\n" + "\n".join(missing)
        if STRICT_PREFLIGHT:
            raise FileNotFoundError(message)
        print("WARNING:", message)
    else:
        print("Preflight OK:", dataset_name)

def load_fixed_subset_bundle(dataset_name: str) -> FixedSubsetBundle:
    csv_path = Path(resolve_dataset_path(dataset_name))
    subset_idx = np.asarray(
        np.load(subset_definition_dir(dataset_name) / "subset_indices_int32.npy", mmap_mode="r"),
        dtype=np.int64,
    )
    manifest = load_json(subset_definition_dir(dataset_name) / "subset_manifest.json")
    if np.any(np.diff(subset_idx) < 0):
        raise RuntimeError("Step-5 subset indices are not sorted.")

    feature_names = read_step5_feature_names(dataset_name)
    context_cols = ["IPV4_SRC_ADDR", "IPV4_DST_ADDR", "PROTOCOL", "Label", "Attack"]
    usecols = list(dict.fromkeys(feature_names + context_cols))

    print(f"\nLoading frozen subset {dataset_name}: {len(subset_idx):,} rows")
    feature_chunks: List[np.ndarray] = []
    context_chunks: Dict[str, List[np.ndarray]] = {c: [] for c in context_cols}
    cursor = 0
    global_start = 0
    t0 = time.time()

    for chunk_no, df_chunk in enumerate(
        pd.read_csv(csv_path, usecols=usecols, chunksize=CSV_CHUNK_SIZE, low_memory=False),
        start=1,
    ):
        global_end = global_start + len(df_chunk)
        left = np.searchsorted(subset_idx, global_start, side="left")
        right = np.searchsorted(subset_idx, global_end, side="left")
        if right > left:
            local_rows = subset_idx[left:right] - global_start
            selected = df_chunk.iloc[local_rows]
            x = np.empty((len(selected), len(feature_names)), dtype=np.float32)
            for j, col in enumerate(feature_names):
                arr = pd.to_numeric(selected[col], errors="coerce").to_numpy(dtype=np.float32, copy=False)
                x[:, j] = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
            feature_chunks.append(x)
            for col in context_cols:
                context_chunks[col].append(selected[col].to_numpy(copy=True))
            cursor += len(selected)
        global_start = global_end
        if chunk_no % 10 == 0 or cursor == len(subset_idx):
            print(f"  chunks={chunk_no} | selected={cursor:,}/{len(subset_idx):,}")
        del df_chunk
        if cursor == len(subset_idx):
            break

    if cursor != len(subset_idx):
        raise RuntimeError(f"Loaded {cursor:,} rows; expected {len(subset_idx):,}.")

    X_raw = np.concatenate(feature_chunks, axis=0)
    context = {k: np.concatenate(v, axis=0) for k, v in context_chunks.items()}
    labels = pd.to_numeric(pd.Series(context["Label"]), errors="coerce").fillna(0).to_numpy(dtype=np.int8)
    attack_codes, attack_classes = pd.factorize(
        pd.Series(context["Attack"]).fillna("<MISSING_ATTACK>").astype(str),
        sort=False,
    )

    print(f"Loaded in {(time.time()-t0)/60:.1f} min | shape={X_raw.shape}")
    return FixedSubsetBundle(
        name=dataset_name,
        feature_names=feature_names,
        X_raw=X_raw,
        src_ip=context["IPV4_SRC_ADDR"],
        dst_ip=context["IPV4_DST_ADDR"],
        protocol=context["PROTOCOL"],
        labels=labels,
        attack_codes=attack_codes.astype(np.int16, copy=False),
        attack_classes=np.asarray(attack_classes, dtype=str),
        subset_global_indices=subset_idx,
        subset_fingerprint=str(manifest["subset_fingerprint"]),
    )

def load_frozen_step5_artifacts(bundle: FixedSubsetBundle):
    shared = step5_shared_dir(bundle.name)
    split_data = np.load(shared / "split_indices.npz")
    train_idx = np.asarray(split_data["train_idx"], dtype=np.int64)
    val_idx = np.asarray(split_data["val_idx"], dtype=np.int64)
    test_idx = np.asarray(split_data["test_idx"], dtype=np.int64)
    split_meta = load_json(shared / "split_metadata.json")
    overlap = load_json(shared / "split_overlap_audit.json")
    scaler = joblib.load(shared / "scaler.joblib")

    if split_meta.get("protocol_name") != PROTOCOL_NAME:
        raise RuntimeError("Step-5 protocol mismatch.")
    if max(
        float(overlap.get("validation_feature_overlap_rate", 1)),
        float(overlap.get("test_feature_overlap_rate", 1)),
    ) != 0.0:
        raise RuntimeError("Frozen Step-5 split has non-zero exact-feature overlap.")
    all_idx = np.concatenate([train_idx, val_idx, test_idx])
    if len(np.unique(all_idx)) != len(all_idx):
        raise RuntimeError("Frozen split row indices overlap.")

    if SMOKE_MODE:
        train_idx = train_idx[:SMOKE_MAX_ROWS_PER_SPLIT]
        val_idx = val_idx[:SMOKE_MAX_ROWS_PER_SPLIT]
        test_idx = test_idx[:SMOKE_MAX_ROWS_PER_SPLIT]

    print(
        f"Frozen split: train={len(train_idx):,}, val={len(val_idx):,}, "
        f"test={len(test_idx):,} | {split_meta['split_fingerprint']}"
    )
    return train_idx, val_idx, test_idx, scaler, split_meta


In [5]:
# Cell 5 — Metrics, validation threshold and operating points

# Cell 8 — Losses, threshold calibration, metrics, and score collection

def focal_bce_with_logits(
    logits: torch.Tensor,
    targets: torch.Tensor,
    alpha: float,
    gamma: float,
    pos_weight: torch.Tensor,
) -> torch.Tensor:
    bce = F.binary_cross_entropy_with_logits(
        logits,
        targets,
        reduction="none",
        pos_weight=pos_weight,
    )
    probabilities = torch.sigmoid(logits)
    pt = torch.where(targets == 1, probabilities, 1 - probabilities)
    alpha_t = torch.where(
        targets == 1,
        torch.as_tensor(alpha, device=targets.device, dtype=targets.dtype),
        torch.as_tensor(1.0 - alpha, device=targets.device, dtype=targets.dtype),
    )
    return (alpha_t * (1 - pt).pow(gamma) * bce).mean()


def safe_divide(numerator: np.ndarray, denominator: np.ndarray) -> np.ndarray:
    result = np.zeros_like(numerator, dtype=np.float64)
    mask = denominator > 0
    result[mask] = numerator[mask] / denominator[mask]
    return result


def find_best_threshold_macro_f1_exact(scores: np.ndarray, labels: np.ndarray) -> Tuple[float, float]:
    scores = np.asarray(scores, dtype=np.float64)
    labels = np.asarray(labels, dtype=np.int64)

    if len(scores) == 0 or len(np.unique(labels)) < 2:
        return 0.0, float(np.median(scores)) if len(scores) else 0.5

    order = np.argsort(scores)
    sorted_scores = scores[order]
    sorted_labels = labels[order]
    unique_ends = np.r_[np.where(np.diff(sorted_scores) != 0)[0], len(sorted_scores) - 1]

    cumulative_positive = np.cumsum(sorted_labels)[unique_ends]
    count_le = unique_ends + 1
    cumulative_negative = count_le - cumulative_positive
    total_positive = int(sorted_labels.sum())
    total_negative = int(len(sorted_labels) - total_positive)

    tp = total_positive - cumulative_positive
    fp = total_negative - cumulative_negative
    fn = cumulative_positive
    tn = cumulative_negative

    f1_positive = safe_divide(2 * tp, 2 * tp + fp + fn)
    f1_negative = safe_divide(2 * tn, 2 * tn + fn + fp)
    macro_f1 = 0.5 * (f1_positive + f1_negative)

    # Candidate: all samples predicted positive.
    all_positive_macro = 0.5 * (
        (2 * total_positive) / max(2 * total_positive + total_negative, 1)
    )

    best_index = int(np.argmax(macro_f1))
    best_f1 = float(macro_f1[best_index])
    best_threshold = float(sorted_scores[unique_ends[best_index]])

    if all_positive_macro > best_f1:
        best_f1 = float(all_positive_macro)
        best_threshold = float(np.nextafter(sorted_scores[0], -np.inf))

    return best_f1, best_threshold




def evaluate_scores(
    collected: Dict[str, np.ndarray],
    threshold: Optional[float] = None,
    score_sign: Optional[int] = None,
) -> Dict[str, Any]:
    raw_scores = collected["scores"].astype(np.float64, copy=False)
    labels = collected["labels"].astype(np.int64, copy=False)

    if len(np.unique(labels)) < 2:
        raise ValueError("Evaluation split must contain both benign and attack labels")

    raw_auc = roc_auc_score(labels, raw_scores)
    if score_sign is None:
        score_sign = -1 if raw_auc < 0.5 else 1
    oriented_scores = score_sign * raw_scores

    auc = roc_auc_score(labels, oriented_scores)
    pr_auc = average_precision_score(labels, oriented_scores)

    if threshold is None:
        f1_macro, threshold = find_best_threshold_macro_f1_exact(oriented_scores, labels)
    else:
        predictions = (oriented_scores > threshold).astype(np.int8)
        f1_macro = f1_score(labels, predictions, average="macro", zero_division=0)

    predictions = (oriented_scores > threshold).astype(np.int8)
    tn, fp, fn, tp = confusion_matrix(labels, predictions, labels=[0, 1]).ravel()

    return {
        "f1_macro": float(f1_macro),
        "auc_roc": float(auc),
        "pr_auc": float(pr_auc),
        "threshold": float(threshold),
        "score_sign": int(score_sign),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "accuracy": float((tp + tn) / max(tp + tn + fp + fn, 1)),
        "precision": float(tp / max(tp + fp, 1)),
        "recall": float(tp / max(tp + fn, 1)),
        "specificity": float(tn / max(tn + fp, 1)),
        "balanced_accuracy": float(
            0.5 * (tp / max(tp + fn, 1) + tn / max(tn + fp, 1))
        ),
        "f1_attack": float(2 * tp / max(2 * tp + fp + fn, 1)),
        "f1_benign": float(2 * tn / max(2 * tn + fn + fp, 1)),
        "fp_rate": float(fp / max(fp + tn, 1)),
        "tp_rate": float(tp / max(tp + fn, 1)),
        "scores": oriented_scores.astype(np.float32),
        "predictions": predictions,
    }


def test_curve_operating_points(scores: np.ndarray, labels: np.ndarray):
    fpr,tpr,rt=roc_curve(labels,scores)
    precision,recall,pt=precision_recall_curve(labels,scores)
    rows=[]
    for target in TEST_CURVE_FPR_TARGETS:
        eligible=np.flatnonzero(fpr<=target+1e-15)
        chosen=eligible[np.argmax(tpr[eligible])] if len(eligible) else 0
        rows.append({'criterion':'TPR_at_test_FPR','target':target,'value':float(tpr[chosen]),'observed':float(fpr[chosen]),'curve_threshold':float(rt[chosen])})
    for target in TEST_CURVE_RECALL_TARGETS:
        eligible=np.flatnonzero(recall>=target-1e-15)
        chosen=eligible[np.argmax(precision[eligible])] if len(eligible) else int(np.argmax(recall))
        thr=float('nan') if chosen>=len(pt) else float(pt[chosen])
        rows.append({'criterion':'Precision_at_test_recall','target':target,'value':float(precision[chosen]),'observed':float(recall[chosen]),'curve_threshold':thr})
    return rows

In [6]:
# Cell 6 — Graph windows and relation-aware THAT-IDS variants
#          (identical to Step 8 except reconstruction_error, marked STEP 9 CHANGE)

@dataclass(frozen=True)
class VariantConfig:
    name: str
    relation_mode: str  # both | ip_ip_only | ip_protocol_only
    drop_protocol_numeric: bool = False

FULL_VARIANT = VariantConfig("full", "both", False)
RELATION_VARIANTS = {
    "ip_ip_only": VariantConfig("ip_ip_only", "ip_ip_only", False),
    "ip_protocol_only": VariantConfig("ip_protocol_only", "ip_protocol_only", False),
    "both_no_protocol_numeric": VariantConfig("both_no_protocol_numeric", "both", True),
}

@dataclass
class GraphWindow:
    edge_feat: torch.Tensor
    labels: torch.Tensor
    src: torch.Tensor
    dst: torch.Tensor
    proto_local: torch.Tensor
    n_ip: int
    n_proto: int
    row_indices: np.ndarray

def protocol_feature_mask(feature_names: List[str], drop_protocol_numeric: bool) -> np.ndarray:
    mask = np.ones(len(feature_names), dtype=bool)
    if not drop_protocol_numeric:
        return mask
    drop_names = {"PROTOCOL", "L7_PROTO"}
    found = []
    for i, name in enumerate(feature_names):
        if str(name).upper() in drop_names:
            mask[i] = False
            found.append(str(name))
    if not found:
        print("WARNING: no PROTOCOL/L7_PROTO numerical columns were found.")
    else:
        print("Dropping numerical protocol fields:", found)
    if not mask.any():
        raise RuntimeError("Feature mask removed every numerical feature.")
    return mask

def permute_split_for_partition(split_idx: np.ndarray, partition_seed: int, split_offset: int) -> np.ndarray:
    split_idx = np.asarray(split_idx, dtype=np.int64)
    if partition_seed == 0:
        return split_idx.copy()
    rng = np.random.default_rng(int(partition_seed) * 1009 + int(split_offset))
    return split_idx[rng.permutation(len(split_idx))]

def build_graph_window(
    bundle: FixedSubsetBundle,
    row_idx: np.ndarray,
    scaler: MinMaxScaler,
) -> GraphWindow:
    raw_feat = bundle.X_raw[row_idx]
    scaled_feat = scaler.transform(raw_feat)
    scaled_feat = np.clip(scaled_feat, CLIP_MIN, CLIP_MAX).astype(np.float32, copy=False)

    src_values = pd.Series(bundle.src_ip[row_idx], copy=False).fillna("<MISSING_IP>").astype(str).to_numpy()
    dst_values = pd.Series(bundle.dst_ip[row_idx], copy=False).fillna("<MISSING_IP>").astype(str).to_numpy()
    proto_values = pd.Series(bundle.protocol[row_idx], copy=False).fillna("<MISSING_PROTOCOL>").astype(str).to_numpy()

    combined_ip = np.concatenate([src_values, dst_values])
    ip_codes, _ = pd.factorize(combined_ip, sort=False)
    n = len(row_idx)
    src = ip_codes[:n].astype(np.int64, copy=False)
    dst = ip_codes[n:].astype(np.int64, copy=False)
    proto_local, proto_uniques = pd.factorize(proto_values, sort=False)
    proto_local = proto_local.astype(np.int64, copy=False)

    return GraphWindow(
        edge_feat=torch.from_numpy(scaled_feat),
        labels=torch.from_numpy(bundle.labels[row_idx].astype(np.int64, copy=False)),
        src=torch.from_numpy(src),
        dst=torch.from_numpy(dst),
        proto_local=torch.from_numpy(proto_local),
        n_ip=int(ip_codes.max() + 1) if len(ip_codes) else 0,
        n_proto=int(len(proto_uniques)),
        row_indices=np.asarray(row_idx, dtype=np.int64),
    )

def prepare_graph_windows(
    bundle: FixedSubsetBundle,
    split_idx: np.ndarray,
    scaler: MinMaxScaler,
    description: str,
) -> List[GraphWindow]:
    windows = []
    n_windows = math.ceil(len(split_idx) / WINDOW_SIZE)
    t0 = time.time()
    for w in range(n_windows):
        idx = split_idx[w * WINDOW_SIZE:(w + 1) * WINDOW_SIZE]
        if len(idx):
            windows.append(build_graph_window(bundle, idx, scaler))
        if (w + 1) % 500 == 0 or w + 1 == n_windows:
            print(f"    {description}: {w + 1}/{n_windows}")
    print(f"  {description}: {len(windows)} batches in {(time.time()-t0)/60:.1f} min")
    return windows

class RelationAttentionEncoder(nn.Module):
    def __init__(self, in_dim: int, out_dim: int, n_relations: int, dropout: float = 0.10):
        super().__init__()
        self.out_dim = out_dim
        self.n_relations = n_relations
        self.proj = nn.Linear(in_dim, out_dim)
        self.edge_attention = nn.ModuleList([
            nn.Linear(out_dim * 2, 1) for _ in range(n_relations)
        ])
        self.semantic_attention = nn.Linear(out_dim, 1)
        self.dropout = nn.Dropout(dropout)
        self.norms = nn.ModuleList([nn.LayerNorm(out_dim) for _ in range(n_relations)])
        self.last_semantic_weights: Optional[torch.Tensor] = None
        self.last_edge_attention_means: Optional[List[float]] = None

    def aggregate_relation(self, h, edge_src, edge_dst, relation_idx):
        if edge_src.numel() == 0:
            self._current_edge_means.append(float("nan"))
            return h
        src_h = h[edge_src]
        dst_h = h[edge_dst]
        alpha = torch.sigmoid(
            self.edge_attention[relation_idx](torch.cat([src_h, dst_h], dim=1))
        )
        self._current_edge_means.append(float(alpha.detach().mean().item()))
        messages = src_h * alpha
        out = torch.zeros_like(h)
        weight_sum = torch.zeros(h.size(0), 1, device=h.device, dtype=h.dtype)
        out.scatter_add_(0, edge_dst.unsqueeze(1).expand(-1, self.out_dim), messages)
        weight_sum.scatter_add_(0, edge_dst.unsqueeze(1), alpha)
        out = out / weight_sum.clamp(min=1.0)
        return self.norms[relation_idx](h + self.dropout(out))

    def forward(self, node_features, relations):
        if len(relations) != self.n_relations:
            raise ValueError(f"Expected {self.n_relations} relations, got {len(relations)}")
        self._current_edge_means = []
        h = F.relu(self.proj(node_features))
        outputs = [
            self.aggregate_relation(h, src, dst, relation_idx)
            for relation_idx, (src, dst) in enumerate(relations)
        ]
        self.last_edge_attention_means = list(self._current_edge_means)
        if len(outputs) == 1:
            weights = torch.ones(
                (1, h.size(0), 1), device=h.device, dtype=h.dtype
            )
            self.last_semantic_weights = weights.detach()
            return outputs[0]
        stacked = torch.stack(outputs, dim=0)
        weights = torch.softmax(self.semantic_attention(stacked), dim=0)
        self.last_semantic_weights = weights.detach()
        return (weights * stacked).sum(dim=0)

class HybridDetector(nn.Module):
    def __init__(self, input_dim, flow_dim, hidden_dim, latent_dim,
                 ctx_recon_weight=CTX_RECON_WEIGHT):
        super().__init__()
        self.flow_dim = flow_dim
        self.ctx_recon_weight = ctx_recon_weight
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.ReLU(), nn.Dropout(ENC_DROPOUT),
            nn.Linear(hidden_dim, latent_dim), nn.ReLU(),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, input_dim)
        )
        self.classifier = nn.Sequential(
            nn.Linear(input_dim + latent_dim, hidden_dim), nn.ReLU(), nn.Dropout(CLS_DROPOUT),
            nn.Linear(hidden_dim, hidden_dim // 2), nn.ReLU(), nn.Linear(hidden_dim // 2, 1),
        )

    def forward(self, x):
        z = self.encoder(x)
        reconstruction = self.decoder(z)
        logits = self.classifier(torch.cat([x, z], dim=1)).squeeze(1)
        return z, reconstruction, logits

    def reconstruction_error(self, x, reconstruction):
        # ---- STEP 9 CHANGE -------------------------------------------------
        # x is the fused input [x_i || c_i]. The first flow_dim columns are the
        # local NetFlow vector and carry no graph gradient. The remaining
        # columns are the gated context block c_i, which IS a function of the
        # graph encoder and of the gate theta.
        #
        # baseline : target kept live  -> gradient can shrink c_i (Step 5/8)
        # detach   : target detached   -> decoder must fit c_i, cannot shrink it
        # drop     : context term off  -> rho = 0
        #
        # Only the TARGET is detached. The prediction (reconstruction) still
        # receives gradient, which is the whole point of the fix.
        mode = globals().get("CONTEXT_TARGET_MODE", "baseline")

        flow_target = x[:, :self.flow_dim]
        ctx_target = x[:, self.flow_dim:]
        if mode in ("detach", "drop"):
            ctx_target = ctx_target.detach()

        flow_err = ((reconstruction[:, :self.flow_dim] - flow_target) ** 2).mean(dim=1)
        if mode == "drop":
            return flow_err
        ctx_err = ((reconstruction[:, self.flow_dim:] - ctx_target) ** 2).mean(dim=1)
        return flow_err + self.ctx_recon_weight * ctx_err
        # ---- END STEP 9 CHANGE ---------------------------------------------

class THATIDSRelationModel(nn.Module):
    def __init__(self, feature_names: List[str], variant: VariantConfig):
        super().__init__()
        self.original_feature_names = list(feature_names)
        self.variant = variant
        mask = protocol_feature_mask(feature_names, variant.drop_protocol_numeric)
        self.register_buffer("feature_mask", torch.from_numpy(mask), persistent=False)
        self.active_feature_names = [n for n, keep in zip(feature_names, mask) if keep]
        self.flow_dim = len(self.active_feature_names)
        n_relations = 2 if variant.relation_mode == "both" else 1

        self.graph_encoder = RelationAttentionEncoder(
            self.flow_dim, HIDDEN_DIM, n_relations, GRAPH_DROPOUT
        )
        self.context_gate = nn.Parameter(torch.tensor(float(INIT_GATE_LOGIT)))
        self.detector = HybridDetector(
            self.flow_dim + HIDDEN_DIM, self.flow_dim, AE_HIDDEN, LATENT_DIM
        )

    def gate_value(self) -> float:
        return float(torch.sigmoid(self.context_gate.detach()).item())

    def prepare_flow_input(self, window: GraphWindow, device: torch.device):
        all_features = window.edge_feat.to(device, non_blocking=True)
        mask = self.feature_mask.to(device)
        flow_features = all_features[:, mask]
        labels = window.labels.to(device, non_blocking=True)
        src = window.src.to(device, non_blocking=True)
        dst = window.dst.to(device, non_blocking=True)
        proto_local = window.proto_local.to(device, non_blocking=True)
        proto = proto_local + window.n_ip
        n_nodes = window.n_ip + window.n_proto
        n_flows = flow_features.size(0)

        node_features = torch.zeros(
            n_nodes, flow_features.size(1), device=device, dtype=flow_features.dtype
        )
        counts = torch.zeros(n_nodes, 1, device=device, dtype=flow_features.dtype)
        ones = torch.ones(n_flows, 1, device=device, dtype=flow_features.dtype)
        for node_ids in (src, dst, proto):
            node_features.scatter_add_(
                0, node_ids.unsqueeze(1).expand(-1, flow_features.size(1)), flow_features
            )
            counts.scatter_add_(0, node_ids.unsqueeze(1), ones)
        node_features = node_features / counts.clamp(min=1.0)

        comm_src = torch.cat([src, dst], dim=0)
        comm_dst = torch.cat([dst, src], dim=0)
        proto_src = torch.cat([src, proto], dim=0)
        proto_dst = torch.cat([proto, src], dim=0)

        if self.variant.relation_mode == "both":
            relations = [(comm_src, comm_dst), (proto_src, proto_dst)]
            readout_nodes = (src, dst, proto)
        elif self.variant.relation_mode == "ip_ip_only":
            relations = [(comm_src, comm_dst)]
            readout_nodes = (src, dst)
        elif self.variant.relation_mode == "ip_protocol_only":
            relations = [(proto_src, proto_dst)]
            readout_nodes = (src, proto)
        else:
            raise ValueError(self.variant.relation_mode)

        node_embeddings = self.graph_encoder(node_features, relations)
        gate = torch.sigmoid(self.context_gate)
        context = gate * sum(node_embeddings[node_ids] for node_ids in readout_nodes) / len(readout_nodes)
        detector_input = torch.cat([flow_features, context], dim=1)
        return detector_input, labels

def load_step5_full_model(dataset_name: str, seed: int, feature_names: List[str]):
    checkpoint = load_torch_checkpoint(
        step5_full_task_dir(dataset_name, seed) / "best.pt", DEVICE
    )
    model = THATIDSRelationModel(feature_names, FULL_VARIANT).to(DEVICE)
    model.load_state_dict(checkpoint["model_state_dict"], strict=True)
    model.eval()
    return model, checkpoint


In [7]:
# Cell 7 — Training, checkpoint/save point and task execution

def focal_bce_with_logits(logits, targets, alpha, gamma, pos_weight):
    bce = F.binary_cross_entropy_with_logits(
        logits, targets, reduction="none", pos_weight=pos_weight
    )
    probabilities = torch.sigmoid(logits)
    pt = torch.where(targets == 1, probabilities, 1 - probabilities)
    alpha_t = torch.where(
        targets == 1,
        torch.as_tensor(alpha, device=targets.device, dtype=targets.dtype),
        torch.as_tensor(1.0-alpha, device=targets.device, dtype=targets.dtype),
    )
    return (alpha_t * (1-pt).pow(gamma) * bce).mean()

def load_history(path: Path) -> List[Dict[str, Any]]:
    return pd.read_csv(path).to_dict(orient="records") if path.exists() else []

def save_history(rows: List[Dict[str, Any]], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(".csv.tmp")
    pd.DataFrame(rows).to_csv(tmp, index=False)
    os.replace(tmp, path)

@torch.no_grad()
def collect_model_components(model, windows, device=DEVICE):
    model.eval()
    probabilities, reconstruction_errors, labels_all = [], [], []
    row_indices = []
    semantic_rows, edge_rows = [], []

    for batch_no, window in enumerate(windows):
        detector_input, labels = model.prepare_flow_input(window, device)
        _, reconstruction, logits = model.detector(detector_input)
        prob = torch.sigmoid(logits)
        rec = model.detector.reconstruction_error(detector_input, reconstruction)

        probabilities.append(prob.cpu().numpy().astype(np.float32))
        reconstruction_errors.append(rec.cpu().numpy().astype(np.float32))
        labels_all.append(labels.cpu().numpy().astype(np.int8))
        row_indices.append(window.row_indices.astype(np.int64))

        weights = model.graph_encoder.last_semantic_weights
        if weights is not None:
            mean_weights = weights.mean(dim=1).squeeze(-1).cpu().numpy()
            for relation_index, value in enumerate(mean_weights):
                semantic_rows.append({
                    "batch": batch_no,
                    "relation_index": relation_index,
                    "weight": float(value),
                })
        edge_means = model.graph_encoder.last_edge_attention_means or []
        for relation_index, value in enumerate(edge_means):
            edge_rows.append({
                "batch": batch_no,
                "relation_index": relation_index,
                "mean_edge_attention": float(value),
            })

    return {
        "probabilities": np.concatenate(probabilities),
        "reconstruction_errors": np.concatenate(reconstruction_errors),
        "labels": np.concatenate(labels_all).astype(np.int64),
        "row_indices": np.concatenate(row_indices).astype(np.int64),
        "semantic_attention": pd.DataFrame(semantic_rows),
        "edge_attention": pd.DataFrame(edge_rows),
    }

def hybrid_scores(probabilities: np.ndarray, reconstruction_errors: np.ndarray, beta: float) -> np.ndarray:
    return (
        float(beta) * probabilities.astype(np.float64)
        + (1.0 - float(beta)) * np.log1p(reconstruction_errors.astype(np.float64))
    )

def train_one_epoch(model, windows, optimizer, positive_class_weight):
    model.train()
    totals = {"loss": 0.0, "reconstruction_loss": 0.0, "classification_loss": 0.0}
    batches = 0
    pos_weight = torch.tensor([positive_class_weight], device=DEVICE, dtype=torch.float32)
    for window in windows:
        detector_input, labels = model.prepare_flow_input(window, DEVICE)
        _, reconstruction, logits = model.detector(detector_input)
        benign_mask = labels == 0
        reconstruction_loss = torch.zeros((), device=DEVICE)
        if benign_mask.any():
            reconstruction_loss = model.detector.reconstruction_error(
                detector_input[benign_mask], reconstruction[benign_mask]
            ).mean()
        classification_loss = focal_bce_with_logits(
            logits, labels.float(), FOCAL_ALPHA, FOCAL_GAMMA, pos_weight
        )
        loss = LAMBDA_RECON * reconstruction_loss + LAMBDA_CLS * classification_loss
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
        optimizer.step()
        totals["loss"] += float(loss.detach().item())
        totals["reconstruction_loss"] += float(reconstruction_loss.detach().item())
        totals["classification_loss"] += float(classification_loss.detach().item())
        batches += 1
    return {k: v / max(batches, 1) for k, v in totals.items()}

def evaluate_component_bundle(collected, threshold=None, score_sign=None):
    scores = hybrid_scores(
        collected["probabilities"], collected["reconstruction_errors"], SCORE_CLS_WEIGHT
    )
    result = evaluate_scores(
        {"scores": scores, "labels": collected["labels"]},
        threshold=threshold,
        score_sign=score_sign,
    )
    return result, scores

def run_training_task(
    bundle: FixedSubsetBundle,
    variant: VariantConfig,
    model_seed: int,
    train_windows: List[GraphWindow],
    val_windows: List[GraphWindow],
    test_windows: List[GraphWindow],
    positive_class_weight: float,
    task_dir: Path,
    experiment_payload: Dict[str, Any],
) -> Dict[str, Any]:
    task_dir.mkdir(parents=True, exist_ok=True)
    completed_path = task_dir / "completed.json"
    best_path = task_dir / "best.pt"
    last_path = task_dir / "last.pt"
    history_path = task_dir / "history.csv"
    result_path = task_dir / "test_metrics.json"
    val_prediction_path = task_dir / "validation_predictions.npz"
    test_prediction_path = task_dir / "test_predictions.npz"
    config_path = task_dir / "config.json"
    fingerprint_path = task_dir / "config_fingerprint.txt"

    payload = {
        **experiment_payload,
        "dataset": bundle.name,
        "subset_fingerprint": bundle.subset_fingerprint,
        "model_seed": int(model_seed),
        "variant": asdict(variant),
        "feature_names": bundle.feature_names,
        "window_size": WINDOW_SIZE,
        "max_epochs": MAX_EPOCHS,
        "patience": PATIENCE,
        "optimizer": "AdamW",
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
    }
    fingerprint = config_fingerprint(payload)

    if completed_path.exists():
        completed = load_json(completed_path)
        if (
            completed.get("config_fingerprint") == fingerprint
            and result_path.exists() and val_prediction_path.exists()
            and test_prediction_path.exists()
        ):
            print("  SKIP completed:", task_dir)
            return load_json(result_path)
        if completed.get("config_fingerprint") != fingerprint:
            raise RuntimeError(f"Fingerprint mismatch in completed task: {task_dir}")
        completed_path.unlink(missing_ok=True)

    if fingerprint_path.exists() and fingerprint_path.read_text().strip() != fingerprint:
        raise RuntimeError(
            f"Config fingerprint mismatch: {task_dir}. "
            "Use a new STEP9_EXPERIMENT_TAG or remove this task directory."
        )
    atomic_json_dump(payload, config_path)
    fingerprint_path.write_text(fingerprint, encoding="utf-8")

    seed_everything(model_seed)
    model = THATIDSRelationModel(bundle.feature_names, variant).to(DEVICE)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )
    parameter_count = sum(p.numel() for p in model.parameters() if p.requires_grad)

    start_epoch = 1
    best_val_f1 = -1.0
    best_epoch = -1
    best_threshold = 0.5
    best_score_sign = 1
    patience_counter = 0
    history = load_history(history_path)

    if last_path.exists():
        checkpoint = load_torch_checkpoint(last_path, DEVICE)
        if checkpoint.get("config_fingerprint") != fingerprint:
            raise RuntimeError(f"Checkpoint fingerprint mismatch: {last_path}")
        model.load_state_dict(checkpoint["model_state_dict"])
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
        start_epoch = int(checkpoint["epoch"]) + 1
        best_val_f1 = float(checkpoint["best_val_f1"])
        best_epoch = int(checkpoint["best_epoch"])
        best_threshold = float(checkpoint["best_threshold"])
        best_score_sign = int(checkpoint.get("best_score_sign", 1))
        patience_counter = int(checkpoint["patience_counter"])
        restore_rng_state(checkpoint.get("rng_state"))
        print(f"  RESUME {variant.name} seed={model_seed} from epoch {start_epoch}")
    else:
        print(f"  START {variant.name} seed={model_seed} | params={parameter_count:,}")

    task_start = time.time()
    for epoch in range(start_epoch, MAX_EPOCHS + 1) if patience_counter < PATIENCE else []:
        epoch_start = time.time()
        train_metrics = train_one_epoch(
            model, train_windows, optimizer, positive_class_weight
        )
        val_collected = collect_model_components(model, val_windows)
        val_metrics, _ = evaluate_component_bundle(val_collected)

        row = {
            "epoch": epoch,
            "train_loss": train_metrics["loss"],
            "train_reconstruction_loss": train_metrics["reconstruction_loss"],
            "train_classification_loss": train_metrics["classification_loss"],
            "val_f1_macro": val_metrics["f1_macro"],
            "val_auc_roc": val_metrics["auc_roc"],
            "val_pr_auc": val_metrics["pr_auc"],
            "val_threshold": val_metrics["threshold"],
            "gate": model.gate_value(),
            "epoch_seconds": time.time() - epoch_start,
        }
        history = [h for h in history if int(h["epoch"]) != epoch]
        history.append(row)
        history.sort(key=lambda x: int(x["epoch"]))
        save_history(history, history_path)

        improved = val_metrics["f1_macro"] > best_val_f1
        if improved:
            best_val_f1 = float(val_metrics["f1_macro"])
            best_epoch = epoch
            best_threshold = float(val_metrics["threshold"])
            best_score_sign = int(val_metrics["score_sign"])
            patience_counter = 0
            atomic_torch_save({
                "model_state_dict": model.state_dict(),
                "epoch": epoch,
                "best_val_f1": best_val_f1,
                "best_threshold": best_threshold,
                "best_score_sign": best_score_sign,
                "variant": asdict(variant),
                "config_fingerprint": fingerprint,
            }, best_path)
        else:
            patience_counter += 1

        if epoch % SAVE_LAST_EVERY == 0 or improved or epoch == MAX_EPOCHS:
            atomic_torch_save({
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "epoch": epoch,
                "best_val_f1": best_val_f1,
                "best_epoch": best_epoch,
                "best_threshold": best_threshold,
                "best_score_sign": best_score_sign,
                "patience_counter": patience_counter,
                "rng_state": capture_rng_state(),
                "variant": asdict(variant),
                "config_fingerprint": fingerprint,
            }, last_path)

        print(
            f"    Ep {epoch:03d}/{MAX_EPOCHS} | loss={train_metrics['loss']:.5f} | "
            f"valF1={val_metrics['f1_macro']:.4f} | PR={val_metrics['pr_auc']:.4f} | "
            f"thr={val_metrics['threshold']:.4f} | gate={model.gate_value():.4f} | "
            f"pat={patience_counter}/{PATIENCE} | {time.time()-epoch_start:.0f}s"
        )
        del val_collected
        if patience_counter >= PATIENCE:
            print("    Early stop at epoch", epoch, "| best epoch", best_epoch)
            break

    if not best_path.exists():
        raise RuntimeError("Best checkpoint was not created: " + str(best_path))
    best = load_torch_checkpoint(best_path, DEVICE)
    model.load_state_dict(best["model_state_dict"])
    model.eval()

    val_collected = collect_model_components(model, val_windows)
    test_collected = collect_model_components(model, test_windows)
    val_metrics, val_scores = evaluate_component_bundle(val_collected)
    test_metrics, test_scores = evaluate_component_bundle(
        test_collected, threshold=best_threshold, score_sign=best_score_sign
    )

    test_rows = test_collected["row_indices"]
    attack_codes = bundle.attack_codes[test_rows]
    per_attack = []
    for code, attack_name in enumerate(bundle.attack_classes):
        mask = (attack_codes == code) & (test_collected["labels"] == 1)
        count = int(mask.sum())
        if count:
            detected = int(test_metrics["predictions"][mask].sum())
            per_attack.append({
                "attack": str(attack_name),
                "count": count,
                "detected": detected,
                "recall": float(detected / count),
            })

    relation_names = (
        ["communication", "protocol"]
        if variant.relation_mode == "both"
        else ["communication"] if variant.relation_mode == "ip_ip_only"
        else ["protocol"]
    )
    semantic = test_collected["semantic_attention"]
    edge_attn = test_collected["edge_attention"]
    attention_summary = {
        "gate_final": model.gate_value(),
        "relation_names": relation_names,
        "semantic_attention": {},
        "edge_attention": {},
    }
    for i, name in enumerate(relation_names):
        s = semantic.loc[semantic["relation_index"] == i, "weight"] if not semantic.empty else pd.Series(dtype=float)
        e = edge_attn.loc[edge_attn["relation_index"] == i, "mean_edge_attention"] if not edge_attn.empty else pd.Series(dtype=float)
        attention_summary["semantic_attention"][name] = {
            "mean": float(s.mean()) if len(s) else float("nan"),
            "std": float(s.std(ddof=1)) if len(s) > 1 else 0.0,
            "n_batches": int(len(s)),
        }
        attention_summary["edge_attention"][name] = {
            "mean": float(e.mean()) if len(e) else float("nan"),
            "std": float(e.std(ddof=1)) if len(e) > 1 else 0.0,
            "n_batches": int(len(e)),
        }
    atomic_json_dump(attention_summary, task_dir / "attention_gate_summary.json")

    np.savez_compressed(
        val_prediction_path,
        scores=val_scores.astype(np.float32),
        labels=val_collected["labels"].astype(np.int8),
        probabilities=val_collected["probabilities"].astype(np.float32),
        reconstruction_errors=val_collected["reconstruction_errors"].astype(np.float32),
        row_indices=val_collected["row_indices"].astype(np.int64),
    )
    np.savez_compressed(
        test_prediction_path,
        scores=test_scores.astype(np.float32),
        predictions=test_metrics["predictions"].astype(np.int8),
        labels=test_collected["labels"].astype(np.int8),
        probabilities=test_collected["probabilities"].astype(np.float32),
        reconstruction_errors=test_collected["reconstruction_errors"].astype(np.float32),
        row_indices=test_collected["row_indices"].astype(np.int64),
        attack_codes=attack_codes.astype(np.int16),
        attack_classes=bundle.attack_classes.astype(str),
    )

    result = {
        "dataset": bundle.name,
        "variant": variant.name,
        "model_seed": model_seed,
        "parameters": parameter_count,
        "best_epoch": best_epoch,
        "validation": {
            k: v for k, v in val_metrics.items()
            if k not in {"scores", "predictions"}
        },
        "test": {
            k: v for k, v in test_metrics.items()
            if k not in {"scores", "predictions"}
        },
        "gate_final": model.gate_value(),
        "per_attack": per_attack,
        "attention_summary": attention_summary,
        "elapsed_seconds_current_session": time.time() - task_start,
        "elapsed_seconds_history": float(sum(float(r.get("epoch_seconds", 0)) for r in history)),
        "config_fingerprint": fingerprint,
        **experiment_payload,
    }
    atomic_json_dump(result, result_path)
    atomic_json_dump({
        "completed": True,
        "completed_at_unix": time.time(),
        "dataset": bundle.name,
        "variant": variant.name,
        "model_seed": model_seed,
        "config_fingerprint": fingerprint,
    }, completed_path)

    print(
        f"  DONE {variant.name} seed={model_seed} | "
        f"F1={result['test']['f1_macro']:.4f} | "
        f"PR={result['test']['pr_auc']:.4f} | "
        f"FPR={100*result['test']['fp_rate']:.4f}%"
    )
    del model, optimizer, val_collected, test_collected
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return result


In [8]:
# Cell 8 — Context-gradient diagnostic
#
# Detaching the TARGET cannot make the gate gradient zero, and it is not supposed
# to. c_i also enters the encoder input x_tilde = [x_i || c_i], so the decoder
# output itself is a function of c_i. Two distinct gradient paths reach theta
# through the reconstruction loss:
#
#   (P) prediction path : c_i -> encoder -> decoder -> x_hat -> loss
#       Legitimate. It asks "pick c_i so the autoencoder reconstructs well".
#       Present in every arm; removing it would mean cutting c_i out of the model.
#
#   (T) target path     : c_i -> target of the context term -> loss
#       This is the degenerate one the supervisor flagged: the loss can fall by
#       shrinking c_i instead of by fitting it. Step 9 removes exactly this path.
#
# So the correct test is NOT "gate gradient is zero". It is: does the notebook's
# own reconstruction_error, under the selected arm, produce the gradient WITHOUT
# path (T)? We answer that by computing both references on one forward pass and
# comparing the live implementation against them.
#
# The size of path (T) relative to the total is also the number that tells us how
# much the published gate value could have been distorted, so we record it.

def context_gradient_diagnostic(dataset_name: str = None,
                                window_index: int = 0) -> Dict[str, Any]:
    dataset_name = dataset_name or DATASETS_TO_RUN[0]
    preflight_step5(dataset_name)
    bundle = load_fixed_subset_bundle(dataset_name)
    train_idx, _, _, scaler, _ = load_frozen_step5_artifacts(bundle)
    windows = prepare_graph_windows(bundle, train_idx[:WINDOW_SIZE * 2], scaler,
                                    "Gradient probe")
    window = windows[window_index]

    seed_everything(MODEL_SEEDS[0])
    model = THATIDSRelationModel(bundle.feature_names, FULL_VARIANT).to(DEVICE)
    model.train()

    detector_input, labels = model.prepare_flow_input(window, DEVICE)
    _, reconstruction, _ = model.detector(detector_input)
    benign = labels == 0
    if not bool(benign.any()):
        raise RuntimeError("No benign flows in this window; try window_index=1")

    x = detector_input[benign]
    r = reconstruction[benign]
    fd = model.detector.flow_dim
    w = model.detector.ctx_recon_weight
    theta = model.context_gate

    flow_err = ((r[:, :fd] - x[:, :fd]) ** 2).mean(dim=1)
    ctx_live = ((r[:, fd:] - x[:, fd:]) ** 2).mean(dim=1)
    ctx_detached = ((r[:, fd:] - x[:, fd:].detach()) ** 2).mean(dim=1)

    loss_live = (flow_err + w * ctx_live).mean()          # Step 5/8 behaviour
    loss_detached = (flow_err + w * ctx_detached).mean()   # target path removed
    loss_flow_only = flow_err.mean()                       # rho = 0

    def grad_of(loss):
        return float(torch.autograd.grad(loss, theta, retain_graph=True)[0]
                     .detach().cpu().reshape(-1)[0])

    g_live = grad_of(loss_live)
    g_detached = grad_of(loss_detached)
    g_flow_only = grad_of(loss_flow_only)
    g_target_path = g_live - g_detached      # the degenerate component

    # what the notebook ACTUALLY computes under the selected arm
    g_actual = grad_of(model.detector.reconstruction_error(x, r).mean())

    expected = {"baseline": g_live, "detach": g_detached, "drop": g_flow_only}[CONTEXT_TARGET_MODE]
    tol = 1e-9 + 1e-6 * abs(expected)
    passed = abs(g_actual - expected) <= tol

    share = abs(g_target_path) / abs(g_live) if abs(g_live) > 0 else float("nan")

    payload = {
        "context_target_mode": CONTEXT_TARGET_MODE,
        "dataset": dataset_name,
        "ctx_recon_weight": float(w),
        "grad_live_target": g_live,
        "grad_detached_target": g_detached,
        "grad_flow_only": g_flow_only,
        "grad_target_path_component": g_target_path,
        "target_path_share_of_live": share,
        "grad_actual_under_arm": g_actual,
        "grad_expected_under_arm": expected,
        "check_passed": bool(passed),
    }

    print(f"mode={CONTEXT_TARGET_MODE}  (rho={w})")
    print(f"  d/dtheta with LIVE target      : {g_live:+.6e}   <- Step 5/8")
    print(f"  d/dtheta with DETACHED target  : {g_detached:+.6e}")
    print(f"  d/dtheta with rho = 0          : {g_flow_only:+.6e}")
    print(f"  degenerate target-path part    : {g_target_path:+.6e} "
          f"({100 * share:.2f}% of the live gradient)")
    print(f"  notebook actually computes     : {g_actual:+.6e}")
    print(f"  expected under this arm        : {expected:+.6e}")
    print(f"  CHECK {'PASSED' if passed else 'FAILED'}")
    if passed:
        print("  Note: a nonzero gate gradient is CORRECT here. c_i feeds the encoder,")
        print("  so the prediction path always reaches theta. Only the target path is removed.")

    model.zero_grad(set_to_none=True)
    del model, windows, bundle
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return payload


# backwards-compatible alias
verify_context_gradient = context_gradient_diagnostic

In [9]:
# Cell 9 — Context-target rerun phase

def run_context_target_phase():
    global CONTEXT_TARGET_MODE

    gradient_checks = []

    for arm in ARMS_TO_RUN:
        CONTEXT_TARGET_MODE = arm
        print("\n" + "=" * 100)
        print(f"CONTEXT-TARGET ARM: {arm}   (rho = "
              f"{0.0 if arm == 'drop' else CTX_RECON_WEIGHT})")
        print("=" * 100)

        # prove the patch is live for this arm before spending GPU time
        check = context_gradient_diagnostic()
        gradient_checks.append(check)
        atomic_json_dump(check, AUDIT_ROOT / f"gradient_check_{arm}.json")
        if not check["check_passed"]:
            raise RuntimeError(
                f"Gradient check failed for arm '{arm}': the reconstruction "
                f"gradient reaching theta does not match the expected reference. "
                "The patch in Cell 6 is not taking effect; do not trust the runs."
            )

        for dataset_name in DATASETS_TO_RUN:
            print("\n" + "-" * 100)
            print(f"ARM {arm} | DATASET {dataset_name}")
            print("-" * 100)
            preflight_step5(dataset_name)
            bundle = load_fixed_subset_bundle(dataset_name)
            train_idx, val_idx, test_idx, scaler, split_meta = \
                load_frozen_step5_artifacts(bundle)

            y_train = bundle.labels[train_idx]
            positive_class_weight = min(
                int((y_train == 0).sum()) / max(int((y_train == 1).sum()), 1),
                POS_WEIGHT_CAP,
            )

            train_windows = prepare_graph_windows(bundle, train_idx, scaler, "Step9 train")
            val_windows = prepare_graph_windows(bundle, val_idx, scaler, "Step9 validation")
            test_windows = prepare_graph_windows(bundle, test_idx, scaler, "Step9 test")

            for model_seed in MODEL_SEEDS:
                run_training_task(
                    bundle=bundle,
                    variant=FULL_VARIANT,
                    model_seed=model_seed,
                    train_windows=train_windows,
                    val_windows=val_windows,
                    test_windows=test_windows,
                    positive_class_weight=positive_class_weight,
                    task_dir=context_task_dir(arm, dataset_name, model_seed),
                    experiment_payload={
                        "experiment": "context_target",
                        "context_target_mode": arm,
                        "effective_ctx_recon_weight":
                            0.0 if arm == "drop" else CTX_RECON_WEIGHT,
                        "relation_variant": "both_relations",
                        "partition_seed": 0,
                        "split_fingerprint": split_meta["split_fingerprint"],
                    },
                )

            update_master_progress("context_target", f"{arm}/{dataset_name}", "completed")
            del train_windows, val_windows, test_windows, bundle
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    pd.DataFrame(gradient_checks).to_csv(
        AUDIT_ROOT / "step9_gradient_checks.csv", index=False)
    update_master_progress("context_target", status="completed")


In [10]:
# Cell 10 — Aggregate Step-9 results and manuscript-ready outputs

def collect_step9_runs() -> pd.DataFrame:
    rows = []
    for arm in ARMS_TO_RUN:
        for dataset_name in DATASETS_TO_RUN:
            for model_seed in MODEL_SEEDS:
                path = context_task_dir(arm, dataset_name, model_seed) / "test_metrics.json"
                if not path.exists():
                    print(f"  MISSING {arm}/{dataset_name}/seed_{model_seed}")
                    continue
                r = load_json(path)
                test = r.get("test", {})
                val = r.get("validation", {})
                rows.append({
                    "context_target_mode": arm,
                    "effective_ctx_recon_weight": 0.0 if arm == "drop" else CTX_RECON_WEIGHT,
                    "dataset": dataset_name,
                    "model_seed": model_seed,
                    "gate_final": r.get("gate_final"),
                    "best_epoch": r.get("best_epoch"),
                    "parameters": r.get("parameters"),
                    "threshold": val.get("threshold"),
                    "f1_macro": test.get("f1_macro"),
                    "pr_auc": test.get("pr_auc"),
                    "roc_auc": test.get("roc_auc"),
                    "fpr_percent": None if test.get("fp_rate") is None
                                   else 100.0 * test["fp_rate"],
                    "tpr": test.get("tp_rate"),
                    "tp": test.get("tp"), "fp": test.get("fp"),
                    "tn": test.get("tn"), "fn": test.get("fn"),
                })
    return pd.DataFrame(rows)


def summarize_step9(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df
    agg = df.groupby(["context_target_mode", "dataset"]).agg(
        n_seeds=("model_seed", "nunique"),
        gate_mean=("gate_final", "mean"), gate_std=("gate_final", "std"),
        gate_min=("gate_final", "min"), gate_max=("gate_final", "max"),
        f1_macro_mean=("f1_macro", "mean"), f1_macro_std=("f1_macro", "std"),
        pr_auc_mean=("pr_auc", "mean"), pr_auc_std=("pr_auc", "std"),
        fpr_percent_mean=("fpr_percent", "mean"), fpr_percent_std=("fpr_percent", "std"),
        tpr_mean=("tpr", "mean"), tpr_std=("tpr", "std"),
    ).reset_index()
    agg["gate_max_abs_shift_from_init"] = agg.apply(
        lambda r: max(abs(r["gate_min"] - INIT_GATE_VALUE),
                      abs(r["gate_max"] - INIT_GATE_VALUE)), axis=1)
    return agg


def build_gate_comparison(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for dataset_name in DATASETS_TO_RUN:
        base = BASELINE_GATE_REFERENCE.get(dataset_name, [])
        rows.append({
            "dataset": dataset_name, "context_target_mode": "published_step5_step8",
            "gate_mean": float(np.mean(base)) if base else None,
            "gate_min": float(np.min(base)) if base else None,
            "gate_max": float(np.max(base)) if base else None,
            "max_abs_shift_from_init":
                max(abs(v - INIT_GATE_VALUE) for v in base) if base else None,
            "n_seeds": len(base),
        })
        for arm in ARMS_TO_RUN:
            sub = df[(df.dataset == dataset_name) & (df.context_target_mode == arm)]
            if sub.empty:
                continue
            g = sub["gate_final"].astype(float)
            rows.append({
                "dataset": dataset_name, "context_target_mode": arm,
                "gate_mean": float(g.mean()), "gate_min": float(g.min()),
                "gate_max": float(g.max()),
                "max_abs_shift_from_init":
                    float(max(abs(g.min() - INIT_GATE_VALUE),
                              abs(g.max() - INIT_GATE_VALUE))),
                "n_seeds": int(sub["model_seed"].nunique()),
            })
    return pd.DataFrame(rows)


def write_step9_latex(summary: pd.DataFrame, path: Path) -> None:
    """Emit a paper-ready table body. Paste into main.tex if the arms differ."""
    lines = [
        "% Step 9 - context-target gradient control, Full variant only.",
        "% Frozen Step-5 rows and split (benchmark seed 2027); 3 model seeds.",
        r"\begin{tabular}{llcccc}",
        r"\toprule",
        r"Dataset & Context target & Gate & F1-macro & PR-AUC & FPR (\%) \\",
        r"\midrule",
    ]
    label = {"baseline": "live (published)", "detach": "detached",
             "drop": r"removed ($\rho=0$)"}
    for dataset_name in DATASETS_TO_RUN:
        for _, r in summary[summary.dataset == dataset_name].iterrows():
            lines.append(
                f"{dataset_name.replace('_V2','-v2')} & "
                f"{label.get(r['context_target_mode'], r['context_target_mode'])} & "
                f"${r['gate_mean']:.6f}$ & "
                f"${r['f1_macro_mean']:.4f}\\pm{r['f1_macro_std']:.4f}$ & "
                f"${r['pr_auc_mean']:.4f}\\pm{r['pr_auc_std']:.4f}$ & "
                f"${r['fpr_percent_mean']:.4f}\\pm{r['fpr_percent_std']:.4f}$ \\\\")
    lines += [r"\bottomrule", r"\end{tabular}"]
    path.write_text("\n".join(lines), encoding="utf-8")


def aggregate_step9():
    print("\n" + "=" * 100)
    print("AGGREGATING STEP 9")
    print("=" * 100)

    runs = collect_step9_runs()
    if runs.empty:
        print("No completed Step-9 runs found; nothing to aggregate.")
        return

    summary = summarize_step9(runs)
    gates = build_gate_comparison(runs)

    runs.to_csv(OUTPUT_ROOT / "step9_context_target_all_runs.csv", index=False)
    summary.to_csv(OUTPUT_ROOT / "step9_context_target_summary.csv", index=False)
    gates.to_csv(AUDIT_ROOT / "step9_gate_comparison.csv", index=False)
    write_step9_latex(summary, OUTPUT_ROOT / "step9_context_target_table.tex")

    def _as_md(frame):
        try:
            return frame.to_markdown(index=False)
        except Exception:
            return "```\n" + frame.to_string(index=False) + "\n```"

    md_lines = [
        "# Step 9 — Context-target gradient control",
        "",
        f"Gate initialization: `theta0 = {INIT_GATE_LOGIT}` -> `g0 = {INIT_GATE_VALUE:.6f}`.",
        "",
        "Frozen Step-5 rows and split (benchmark seed 2027), Full variant only,",
        f"model seeds {MODEL_SEEDS}. Only the context-target gradient path differs.",
        "",
        "## Gate comparison",
        "",
        _as_md(gates),
        "",
        "## Frozen-benchmark metrics",
        "",
        _as_md(summary),
        "",
        "## How to read this",
        "",
        "If the `detach` gate stays inside the published 0.1817-0.1853 band, the",
        "non-detached target was not what pinned the gate: the paper keeps its",
        "current numbers and states this explicitly. If it moves materially, the",
        "main results must be regenerated from the `detach` arm and the gate",
        "discussion rewritten.",
    ]
    (OUTPUT_ROOT / "step9_summary.md").write_text("\n".join(md_lines), encoding="utf-8")

    manifest = {
        "experiment_tag": STEP9_EXPERIMENT_TAG,
        "arms": ARMS_TO_RUN,
        "datasets": DATASETS_TO_RUN,
        "model_seeds": MODEL_SEEDS,
        "benchmark_split_seed": BENCHMARK_SPLIT_SEED,
        "protocol": PROTOCOL_NAME,
        "variant": "full",
        "ctx_recon_weight_baseline": CTX_RECON_WEIGHT,
        "lambda_recon": LAMBDA_RECON,
        "lambda_cls": LAMBDA_CLS,
        "init_gate_logit": INIT_GATE_LOGIT,
        "init_gate_value": INIT_GATE_VALUE,
        "window_size": WINDOW_SIZE,
        "max_epochs": MAX_EPOCHS,
        "patience": PATIENCE,
        "completed_runs": int(len(runs)),
        "expected_runs": len(ARMS_TO_RUN) * len(DATASETS_TO_RUN) * len(MODEL_SEEDS),
        "published_gate_reference": BASELINE_GATE_REFERENCE,
        "change_relative_to_step8":
            "HybridDetector.reconstruction_error detaches the context block of the "
            "target when CONTEXT_TARGET_MODE is 'detach' or 'drop'; 'drop' also sets "
            "the context weight to zero. Nothing else differs from Step 5/8.",
    }
    atomic_json_dump(manifest, OUTPUT_ROOT / "step9_manifest.json")

    bundle_path = OUTPUT_ROOT / "THAT_IDS_step9_context_target_bundle.zip"
    with zipfile.ZipFile(bundle_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for pattern in ("*.csv", "*.json", "*.md", "*.tex"):
            for f in sorted(OUTPUT_ROOT.glob(pattern)):
                zf.write(f, f.name)
            for f in sorted(AUDIT_ROOT.glob(pattern)):
                zf.write(f, f"audit/{f.name}")
        for arm in ARMS_TO_RUN:
            for dataset_name in DATASETS_TO_RUN:
                for model_seed in MODEL_SEEDS:
                    d = context_task_dir(arm, dataset_name, model_seed)
                    for name in ("test_metrics.json", "config.json",
                                 "config_fingerprint.txt", "history.csv"):
                        f = d / name
                        if f.exists():
                            zf.write(f, f"runs/{arm}/{dataset_name}/seed_{model_seed}/{name}")

    print("\n=== GATE COMPARISON ===")
    display(gates)
    print("\n=== FROZEN-BENCHMARK METRICS ===")
    display(summary)
    print(f"\nWritten to: {OUTPUT_ROOT}")
    print(f"Bundle: {bundle_path}")
    update_master_progress("aggregate", status="completed")


In [11]:
# Cell 11 — RUN ALL orchestrator

def run_selected_phases():
    started = time.time()
    update_master_progress("master", status="running", details={
        "run_mode": RUN_MODE, "arms": ARMS_TO_RUN,
    })

    if RUN_MODE in {"rerun", "all"}:
        run_context_target_phase()

    if RUN_MODE in {"aggregate", "all"}:
        aggregate_step9()

    update_master_progress("master", status="completed", details={
        "run_mode": RUN_MODE,
        "elapsed_hours": (time.time() - started) / 3600.0,
    })
    print(f"\nStep 9 completed for RUN_MODE={RUN_MODE}.")
    print(f"Elapsed: {(time.time() - started) / 3600:.2f} hours")
    print("Output:", OUTPUT_ROOT)

run_selected_phases()



CONTEXT-TARGET ARM: detach   (rho = 0.1)
Preflight OK: CIC_V2

Loading frozen subset CIC_V2: 496,384 rows
  chunks=10 | selected=131,692/496,384
  chunks=20 | selected=263,031/496,384
  chunks=30 | selected=394,663/496,384
  chunks=38 | selected=496,384/496,384
Loaded in 1.7 min | shape=(496384, 39)
Frozen split: train=361,202, val=40,519, test=94,663 | 58cfcf6084a427575c09
    Gradient probe: 2/2
  Gradient probe: 2 batches in 0.0 min
mode=detach  (rho=0.1)
  d/dtheta with LIVE target      : +5.270344e-03   <- Step 5/8
  d/dtheta with DETACHED target  : +2.140238e-04
  d/dtheta with rho = 0          : +2.254969e-04
  degenerate target-path part    : +5.056320e-03 (95.94% of the live gradient)
  notebook actually computes     : +2.140238e-04
  expected under this arm        : +2.140238e-04
  CHECK PASSED
  Note: a nonzero gate gradient is CORRECT here. c_i feeds the encoder,
  so the prediction path always reaches theta. Only the target path is removed.

------------------------------

,dataset,context_target_mode,gate_mean,gate_min,gate_max,max_abs_shift_from_init,n_seeds
0,CIC_V2,published_step5_step8,0.185110,0.184949,0.185217,0.002791,3
1,CIC_V2,detach,0.185924,0.185715,0.186041,0.003615,3
2,CIC_V2,drop,0.185947,0.185758,0.186053,0.003628,3
3,UNSW_V2,published_step5_step8,0.184268,0.184123,0.184362,0.001936,3
4,UNSW_V2,detach,0.185025,0.184952,0.185096,0.002670,3
5,UNSW_V2,drop,0.185047,0.184970,0.185117,0.002692,3



=== FROZEN-BENCHMARK METRICS ===


,context_target_mode,dataset,n_seeds,gate_mean,gate_std,gate_min,gate_max,f1_macro_mean,f1_macro_std,pr_auc_mean,pr_auc_std,fpr_percent_mean,fpr_percent_std,tpr_mean,tpr_std,gate_max_abs_shift_from_init
0,detach,CIC_V2,3,0.185924,0.000181,0.185715,0.186041,0.972880,0.001101,0.983809,0.001130,0.023185,0.004250,0.920611,0.003283,0.003615
1,detach,UNSW_V2,3,0.185025,0.000072,0.184952,0.185096,0.994802,0.002128,0.997349,0.000264,0.609775,0.235361,0.999212,0.000676,0.002670
2,drop,CIC_V2,3,0.185947,0.000164,0.185758,0.186053,0.972420,0.000533,0.984007,0.000955,0.019939,0.002125,0.919191,0.001466,0.003628
3,drop,UNSW_V2,3,0.185047,0.000074,0.184970,0.185117,0.994939,0.001911,0.997335,0.000295,0.595687,0.210970,0.999280,0.000627,0.002692



Written to: /content/drive/MyDrive/THAT_IDS_RERUN/conference_step9_context_target_v1
Bundle: /content/drive/MyDrive/THAT_IDS_RERUN/conference_step9_context_target_v1/THAT_IDS_step9_context_target_bundle.zip

Step 9 completed for RUN_MODE=all.
Elapsed: 0.25 hours
Output: /content/drive/MyDrive/THAT_IDS_RERUN/conference_step9_context_target_v1
